In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import jax
import jax.numpy as jnp

from pixelpop.models.gwpop_models import trunc_gaussian

/home/noah.wolfe/.conda/envs/just-for-kicks/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np

def build_interp_sampler(density, xs, xp=jnp):
    """ factory-function for inverse CDF sampling by interpolated a density
        over points xp. """
    if xp == jnp:
        from quadax import cumulative_trapezoid
    else:
        from scipy.integrate import cumulative_trapezoid

    prob = density(xs)
    norm = xp.trapezoid(prob, xs)
    prob /= norm

    cdf = cumulative_trapezoid(y=prob, x=xs, initial=0)

    if xp == jnp:
        def func(key):
            u = jax.random.uniform(key)
            return xp.interp(u, cdf, xs)
    elif xp == np:
        def func(rng, size=()):
            u = rng.uniform(size=size)
            return xp.interp(u, cdf, xs)

    return func

In [3]:
def event_log_likelihood(x, data, sigma_n=1.0):
    return -(x - data)**2 / 2 / sigma_n**2 - 2 * jnp.log(2 * jnp.pi * sigma_n**2)

In [4]:
def log_model(x, mu_x, sigma_x):
    return trunc_gaussian(x, mean=mu_x, sig=sigma_x, lower=-1, upper=1)

In [5]:
from util import logtrapz

def log_marg_likelihood(data, mu_x, sigma_x, sigma_n=1.0):
    xs = jnp.linspace(-1, 1, 100)
    ll = event_log_likelihood(xs, data, sigma_n=sigma_n)
    lp = log_model(xs, mu_x, sigma_x)
    return logtrapz(ll + lp, xs)

In [8]:
def catalog_log_likelihood(data, mu_x, sigma_x, sigma_n=1.0):
    ll = jax.vmap(
        lambda d: log_marg_likelihood(d, mu_x, sigma_x, sigma_n=sigma_n)
    )(data)
    return jnp.sum(ll)

In [6]:
mu_x = 0
sigma_x = 1.0
sigma_n = 0.01

x = jax.random.normal(jax.random.key(1)) * sigma_x
n = jax.random.normal(jax.random.key(2)) * sigma_n

d = x + n

event_log_likelihood(x, d, sigma_n=sigma_n)

Array(14.679919, dtype=float32)

In [7]:
log_marg_likelihood(d, mu_x, sigma_x, sigma_n=sigma_n)

Array(10.525431, dtype=float32)